# Prophet + DTW neighbours (N ∈ {2, 3, 5, 7, 10, 20})

Upstream source: `Dtw_prophet_N.ipynb` in the thesis workbench.

**Input**: `DTW_DFS_DIR / 'dtw_{N}_dfs' / *.parquet` (per-keyword panels with top-N DTW neighbours).
**Output**: `prophet_results_dir('dtw_neighbour_dfs/prophet_results_multiN_dtw')` — `univariate_metrics.csv`, `multivariate_metrics_allN.csv`, per-keyword forecast CSVs under `univariate/forecasts/` and `N{N}/multivariate/forecasts/`.


In [ ]:
# --- Paper-repo bootstrap (replaces original Colab `drive.mount` cell) ---
import sys, os
from pathlib import Path

_REPO_SHARED = Path("..", "_shared").resolve()
if str(_REPO_SHARED) not in sys.path:
    sys.path.insert(0, str(_REPO_SHARED))

from paths import ensure_env, DATA_ROOT, DTW_DFS_DIR, KEYWORDS_DIR_5, prophet_results_dir  # noqa: E402

ensure_env()


In [ ]:
# --- Core imports ---
from pathlib import Path
from datetime import date
import numpy as np
import pandas as pd
import polars as pl
from tqdm.auto import tqdm
from prophet import Prophet

# ---------------- Config ----------------
BASE = DTW_DFS_DIR
N_LIST = [2, 3, 5, 7, 10, 20]
KW_DIRS = {N: BASE / f"dtw_{N}_dfs" for N in N_LIST}

OUT_DIR = BASE / "prophet_results_multiN_dtw"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# metrics files
UNI_METRICS_CSV   = OUT_DIR / "univariate_metrics.csv"
MULTI_METRICS_CSV = OUT_DIR / "multivariate_metrics_allN.csv"

TEST_WEEKS = 6

# ---------------- Helpers ----------------
def iso_week_to_date(wwyyyy: str) -> date:
    w, y = wwyyyy.split("-")
    return date.fromisocalendar(int(y), int(w), 1)

def smape(y_true, y_pred):
    """
    Symmetric Mean Absolute Percentage Error (SMAPE)
    Defined as:
        SMAPE = 100% * (1/n) * Σ( |y_pred - y_true| / ((|y_true| + |y_pred|) / 2) )
    Handles zero denominators safely.
    """
    y_true, y_pred = np.array(y_true, dtype=float), np.array(y_pred, dtype=float)
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    mask = denom != 0
    smape_val = np.mean(np.abs(y_pred[mask] - y_true[mask]) / denom[mask])
    return 100 * smape_val


def rmse(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return float(np.sqrt(np.mean((y_pred - y_true)**2)))

def make_prophet(add_regressors=None):
    m = Prophet(
        daily_seasonality=False,
        weekly_seasonality=True,
        yearly_seasonality=True,
        seasonality_mode="additive",
        changepoint_prior_scale=0.05,
        interval_width=0.8,
        stan_backend="CMDSTANPY",
    )
    if add_regressors:
        for r in add_regressors:
            m.add_regressor(r, standardize=True)
    return m

def prep_prophet_frames(pl_df: pl.DataFrame, use_exogenous: bool, exog_cols=None):
    base = (
        pl_df
        .select(pl.col("week"), pl.col("cpc_week").cast(pl.Float64).alias("y"))
        .with_columns(pl.col("week").map_elements(iso_week_to_date, return_dtype=pl.Date).alias("ds"))
        .drop("week")
        .sort("ds")
    ).filter(pl.col("y").is_not_null())

    if not use_exogenous:
        return base.to_pandas(), []

    if not exog_cols:
        return base.to_pandas(), []

    exog = (
        pl_df
        .select(["week"] + exog_cols)
        .with_columns(pl.col("week").map_elements(iso_week_to_date, return_dtype=pl.Date).alias("ds"))
        .drop("week")
        .sort("ds")
    )

    full = base.join(exog, on="ds", how="left")
    pdf = full.to_pandas()
    pdf[exog_cols] = pdf[exog_cols].ffill().bfill()
    return pdf, exog_cols


def train_eval_save(pdf, reg_cols, out_fc_dir, out_name):
    if len(pdf) <= TEST_WEEKS + 5:
        return np.nan, np.nan

    train = pdf.iloc[:-TEST_WEEKS]
    test = pdf.iloc[-TEST_WEEKS:]

    m = make_prophet(add_regressors=reg_cols)
    m.fit(train)

    fc = m.predict(test[["ds"] + reg_cols] if reg_cols else test[["ds"]])

    out_fc_dir.mkdir(parents=True, exist_ok=True)
    pd.DataFrame({
        "ds": test["ds"],
        "y_true": test["y"],
        "yhat": fc["yhat"],
    }).to_csv(out_fc_dir / f"{out_name}.csv", index=False)

    return smape(test["y"], fc["yhat"]), rmse(test["y"], fc["yhat"])

# ---------------------------------------------------------
# 1) RUN UNIVARIATE ONLY ONCE (using N=20 dataset)
# ---------------------------------------------------------
print("Running UNIVARIATE once on N=20 dataset...")
uni_metrics = []

files_20 = sorted(KW_DIRS[20].glob("*.parquet"))
uni_fc_dir = OUT_DIR / "univariate" / "forecasts"
uni_fc_dir.mkdir(parents=True, exist_ok=True)

for p in tqdm(files_20):
    kw_slug = p.stem
    df_pl = pl.read_parquet(p)

    df_pl = df_pl.select([c for c in df_pl.columns if c in ("week", "cpc_week")])
    pdf_uni, reg_uni = prep_prophet_frames(df_pl, use_exogenous=False)
    s_uni, r_uni = train_eval_save(pdf_uni, [], uni_fc_dir, kw_slug)
    uni_metrics.append({
        "keyword": kw_slug,
        "smape": s_uni,
        "rmse": r_uni,
    })

pd.DataFrame(uni_metrics).to_csv(UNI_METRICS_CSV, index=False)

print("Univariate done.")

# ---------------------------------------------------------
# 2) MULTIVARIATE for each N
# ---------------------------------------------------------
multi_metrics_all = []

for N in N_LIST:
    print(f"\n=== Running MULTIVARIATE for N={N} ===")

    in_dir = KW_DIRS[N]
    multi_fc_dir = OUT_DIR / f"N{N}" / "multivariate" / "forecasts"
    multi_fc_dir.mkdir(parents=True, exist_ok=True)

    files = sorted(in_dir.glob("*.parquet"))

    for p in tqdm(files, desc=f"N={N}"):
        kw_slug = p.stem
        df_pl = pl.read_parquet(p)

        # include all numeric exogenous features except the ads metrics
        drop_prefixes = ("adcost", "adclick", "adclocks")
        num_cols = [c for c, t in zip(df_pl.columns, df_pl.dtypes)
                    if t in (pl.Float32, pl.Float64, pl.Int32, pl.Int64, pl.UInt32, pl.UInt64)
                    and c not in ("week", "cpc_week", "year", "week_num")
                    and not c.startswith(drop_prefixes)]
        df_pl = df_pl.select(["week", "cpc_week"] + num_cols)


        pdf, reg_cols = prep_prophet_frames(df_pl, use_exogenous=True, exog_cols=num_cols)
        s, r = train_eval_save(pdf, reg_cols, multi_fc_dir, kw_slug)

        multi_metrics_all.append({
            "keyword": kw_slug,
            "n_neighbors": N,
            "smape": s,
            "rmse": r,
        })

pd.DataFrame(multi_metrics_all).to_csv(MULTI_METRICS_CSV, index=False)

print("All done.")


In [ ]:
# --- Interpretation: aggregate metrics by number of neighbours ---
import pandas as pd
import numpy as np

# Read saved metrics
uni = pd.read_csv(prophet_results_dir("dtw_neighbour_dfs/prophet_results_multiN_dtw") / "univariate_metrics.csv")
multi = pd.read_csv(prophet_results_dir("dtw_neighbour_dfs/prophet_results_multiN_dtw") / "multivariate_metrics_allN.csv")

# Tag univariate as "0 neighbours"
uni_agg = uni.assign(n_neighbors=0)[["keyword", "n_neighbors", "smape", "rmse"]]

# Combine & keep valid rows
allm = pd.concat(
    [uni_agg, multi[["keyword", "n_neighbors", "smape", "rmse"]]],
    ignore_index=True
)

# Ensure numeric + drop missing metrics
allm["smape"] = pd.to_numeric(allm["smape"], errors="coerce")
allm["rmse"]  = pd.to_numeric(allm["rmse"], errors="coerce")

summary = (
    allm.dropna(subset=["smape", "rmse"])
       .groupby("n_neighbors")
       .agg(
           n_keywords=("keyword", "nunique"),
           smape_mean=("smape", "mean"),
           smape_median=("smape", "median"),
           rmse_mean=("rmse", "mean"),
           rmse_median=("rmse", "median"),
       )
       .reset_index()
       .sort_values("n_neighbors")
)

# Nice formatting for display
summary_display = summary.copy()
for c in ["smape_mean", "smape_median", "rmse_mean", "rmse_median"]:
    summary_display[c] = summary_display[c].round(4)

summary_display
